In [3]:
# --- PAIRWISE CROSS-ENCODER FEATURES ---
# Logic: Run pre-trained cross-encoders over every item *pair* to extract
#        sentiment, NLI (entail / contradict / neutral) and semantic similarity
#        scores; derive a handful of interaction features on top.
# Workflow: <split>_item_correlations.parquet -> per-pair features ->
#           <split>_item_correlations_with_cross.parquet.

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import polars as pl
import torch
from sentence_transformers import CrossEncoder

# ---- Config ----

# Which split to score: "" (training), "holdout_" or "validation_"
splits = ["validation_", "holdout_", ""]
for split in splits:
    batch_size   = 512
    parquet_path = f"../../data/raw/{split}item_correlations.parquet"
    output_path  = f"../../data/processed/{split}item_correlations_with_cross.parquet"

    os.makedirs("../../data/processed", exist_ok=True)

    # ---- Load pair table ----

    dat = pl.read_parquet(parquet_path)
    pairs = dat.select(["Parameter1", "Parameter2"]).to_numpy().tolist()

    # ---- Score pairs through each cross-encoder ----

    # Sentiment + NLI return softmaxed class probs; the reranker returns a scalar.
    models_to_run = [
        #{"name": "cardiffnlp/twitter-roberta-base-sentiment-latest", "nick": "sentiment"},
        {"name": "cross-encoder/nli-deberta-v3-base",                "nick": "nli"},
        {"name": "mixedbread-ai/mxbai-rerank-base-v1",              "nick": "similarity"},
    ]

    for m in models_to_run:
        print(f"\n--- Loading {m['name']} ---")
        model = CrossEncoder(m["name"], device="cuda", model_kwargs={"torch_dtype": torch.float16})

        if m["nick"] in ("nli", "sentiment"):
            # Softmax over the class logits so columns are calibrated probabilities
            raw = model.predict(pairs, batch_size=batch_size, show_progress_bar=True, apply_softmax=True)

            if m["nick"] == "nli":
                dat = dat.with_columns([
                    pl.Series("contradiction", raw[:, 0]),
                    pl.Series("entail",        raw[:, 1]),
                    pl.Series("logic_neutral", raw[:, 2]),
                ])
            else:
                # CardiffNLP order: negative, neutral, positive (skip neutral here)
                dat = dat.with_columns([
                    pl.Series("pair_negative", raw[:, 0]),
                    pl.Series("pair_positive", raw[:, 2]),
                ])
        else:
            # Scalar reranker score
            scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=True)
            dat = dat.with_columns(pl.Series(m["nick"], scores))

        del model
        torch.cuda.empty_cache()

    # ---- Derived interaction features ----

    dat = dat.with_columns([
        # Similarity weighted by entailment - high when items overlap and agree
        (pl.col("similarity") * pl.col("entail")).alias("thematic_intensity"),
        # Similarity weighted by contradiction - high when items overlap but disagree
        (pl.col("similarity") * pl.col("contradiction")).alias("logical_friction"),
        # Net pair-level sentiment polarity
        #(pl.col("pair_positive") - pl.col("pair_negative")).alias("sentiment_balance"),
    ])

    # ---- Save ----

    dat.write_parquet(output_path)
    print(f"Done! Saved to {output_path}")



--- Loading cross-encoder/nli-deberta-v3-base ---


Batches: 100%|██████████| 171/171 [00:39<00:00,  4.38it/s]



--- Loading mixedbread-ai/mxbai-rerank-base-v1 ---


Batches: 100%|██████████| 171/171 [00:32<00:00,  5.25it/s]


Done! Saved to ../../data/processed/holdout_item_correlations_with_cross.parquet

--- Loading cross-encoder/nli-deberta-v3-base ---


Batches: 100%|██████████| 893/893 [03:01<00:00,  4.92it/s]



--- Loading mixedbread-ai/mxbai-rerank-base-v1 ---


Batches: 100%|██████████| 893/893 [03:00<00:00,  4.95it/s]


OSError: Der Vorgang ist bei einer Datei mit einem geöffneten Bereich, der einem Benutzer zugeordnet ist, nicht anwendbar. (os error 1224): ../../data/processed/item_correlations_with_cross.parquet